# Clean the Table with Coupled Newton Solvers


The migration notebook kept the two-cube stacking task familiar. This exercise adds materials that bend and fold: use the **gripper** to pick up two cubes, a free cloth patch and a cable, carry each to the receiving box, and release it inside.

**SolverMuJoCo** owns the robot and both cubes. **SolverVBD** owns the cable bodies/rod joints and the cloth particles. **SolverCoupledProxy** exposes both jaws and relevant cube bodies to the material solver and returns contact reactions to the rigid solver.

The cloth is a small, single-layer shirt silhouette with an initial fold, not a sewn garment. Each pickup must come from opposing gripper contacts; no payload is attached to a jaw or moved by a scripted pose assignment.

This notebook runs the selected implementation and checks each object’s grasp, lift, carry, release and settling. See the [verification record](../VERIFICATION.md) for tested robot/device combinations. Full CPU runs can take tens of minutes; the rollout cell displays captured output only after completion.

## Outline

- Build an open box and define its usable interior.
- Add a free cloth mesh and a capsule rod.
- Assign disjoint solver ownership and connect both gripper jaws.
- Advance contacts and dynamics through one coupled loop.
- Measure each grasp, full lift, carry, opening and detached placement.

## What you will learn

You will connect Newton's model/state/control workflow to a manipulation task with rigid and deformable objects. The key decisions are who integrates each object, where forces cross the solver boundary, and which measured events distinguish a pickup from an object merely ending up inside a box.

## Setup: check the environment

Use the dedicated Python 3.12 environment in the [Newton setup guide](../README.md). It pins Newton 1.6.0, Warp 1.17.0, MuJoCo/MuJoCo Warp 3.12.0, newton-usd-schemas 0.5.0 and usd-core 26.3. The optional `requirements.lock.txt` fixes all resolved dependency versions and hashes. VBD and coupled solvers are experimental.

From `tutorials/newton`, launch Jupyter with the OpenUSD Work limit already set:

```bash
PXR_WORK_THREAD_LIMIT=1 .venv/bin/jupyter lab --notebook-dir notebooks
```

Select the Newton kernel and restart any existing kernel after changing the launch environment. On PowerShell, use `$env:PXR_WORK_THREAD_LIMIT="1"` before launching. The usd-core 26.3 helper requires this setting because parallel robot-USD imports failed natively on macOS and Linux; a late cell assignment after `pxr` loads is insufficient. Windows has not been validated.

`REFERENCE = True` selects the reference implementations directly. Set it to `False` after completing your exercise files. Reference mode neither fills TODOs nor overwrites your work, and selecting it is not a claim that a task has passed.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import subprocess
import json
import shutil
import numpy as np
import newton
import warp as wp

PART = Path.cwd()
if not (PART / "clean_table_task.py").exists():
    PART = PART / "newton" / "part3"
assert (PART / "clean_table_task.py").exists(), "Open this notebook from newton/part3 or the notebooks directory"
assert newton.__version__ == "1.6.0", "Install the pinned Article_3 requirements in this kernel"
REFERENCE = True
ROBOT = "so101"  # also "rebot"
wp.init()
newton.use_coord_layout_targets = True  # Already the 1.6 default; explicit before authoring.
print("Python:", sys.executable)
for package in ("newton", "warp-lang", "mujoco", "mujoco-warp", "newton-usd-schemas", "usd-core"):
    print(f"{package}: {version(package)}")
from pxr import Usd, Work
print("OpenUSD:", Usd.GetVersion(), "Work concurrency:", Work.GetConcurrencyLimit())
if Usd.GetVersion() == (0, 26, 3):
    assert Work.GetConcurrencyLimit() == 1, "Restart Python/Jupyter with PXR_WORK_THREAD_LIMIT=1"
print("Task device: cpu; optional CUDA and GL checks remain separate")
print("Running", "reference solutions" if REFERENCE else "your exercise files")


## Overview of the files

| File | Your work |
|------|-----------|
| `clean_table_scene.py` | Steps 0–3: bin, cloth, cable and solver coupling |
| `clean_the_table.py` | Step 4: the out-of-place coupled substep loop |
| `clean_table_task.py` | Supplied bounded gripper IK, target mapping, geometry and measured grasp-history helpers |
| `newton_assets.py` | Pinned official robot USD import with Newton/Mjc schema resolvers |
| `solutions/clean_table_scene_solution.py` | Reference scene and solver factories |
| `solutions/clean_the_table_solution.py` | Reference sequential gripper controller and runner |
| `solutions/clean_step_*.py` | Annotated functions for the exercises |
| `final_check.py` | Shared pure report gate used after the rollout |

Keep the two exercise files beside the notebook. The next cell creates backups only if none exist.

The dynamic robot is an official structured USD imported by `add_robot_usd`. Keep its geometry and physics layers together. The helper registers the explicit schema mapping needed for passive damping, collision filters, actuators and gripper constraints. The separate Menagerie MJCF is a scratch robot model for IK and reBot gravity feedforward; changing its pose does not move the simulated payload.

Automatic asset caches must match their pinned revision and have unchanged tracked robot files. If validation fails, preserve the directory and select a fresh `NEWTON_CACHE_PATH` or `MUJOCO_MENAGERIE_CACHE`, as described in the setup guide. Explicit Menagerie path overrides are available for deliberate custom models and do not replace the USD robot.

Task servo gains and the SO-101 arm-force policy are authored in the scene after import. Those settings are simulation choices, not claims about physical hardware.

In [ ]:
for name in ("clean_table_scene.py", "clean_the_table.py"):
    source = PART / name
    backup = source.with_name(source.stem + "_original.py")
    if not backup.exists():
        shutil.copy2(source, backup)
    print("Backup preserved:", backup.name)


## Baseline: recognize an unfinished exercise

A starter should raise one of the five exact `NotImplementedError: Complete TODO Step` errors below. Construction can reach the cable exercise before the cloth exercise; follow the error naming the next unfinished function. These known TODOs do not block reference mode. A dependency, syntax, cache or unrelated runtime failure must still raise. If your starter is complete, this one-frame probe can succeed; it does not establish a completed pickup.

In [ ]:
try:
    baseline = subprocess.run(
        [sys.executable, "clean_the_table.py", "--device", "cpu", "--viewer", "null", "--num-frames", "1"],
        cwd=PART, text=True, capture_output=True, timeout=180, check=True,
    )
except subprocess.CalledProcessError as exc:
    expected = {
        "NotImplementedError: Complete TODO Step 0: add_bin",
        "NotImplementedError: Complete TODO Step 1: add_garment",
        "NotImplementedError: Complete TODO Step 2: add_cable",
        "NotImplementedError: Complete TODO Step 3: build_coupled_solver",
        "NotImplementedError: Complete TODO Step 4: coupled substep loop",
    }
    last_line = (exc.stderr or "").strip().splitlines()[-1:]
    if not last_line or last_line[0] not in expected:
        print(exc.stdout or "")
        print(exc.stderr or "")
        raise
    print("EXPECTED incomplete-exercise error:", last_line[0])
else:
    print("The starter already runs; continue with its validation.")


### 0. Build the bin

At `TODO Step 0`, complete `add_bin` using **five static boxes**: a floor and four walls. Attach them to `body=-1` so they belong to the world. Zero density alone does not select the static attachment.

Return the shape IDs and a `Bin` whose lower and upper coordinates describe the inner faces. The gripper must lift each complete object over the rim and open above that interior. The bounds define the usable receiving region, including clearance for cloth and cable, rather than the outside dimensions of the rendered box.

Compare with `solutions/clean_step_00_add_bin.py` if needed.

### 1. Add the garment

At `TODO Step 1`, complete `add_garment` with `builder.add_cloth_mesh`. The reference authors a small triangulated shirt silhouette with an elastic crease forming a raised initial fold that the jaws can approach. Every particle has positive mass; no cloth point is pinned to a jaw or the table. Initial geometry should avoid penetrations while leaving a physically reachable grasp region.

Membrane stiffness (`tri_ke`, `tri_ka`), bending stiffness (`edge_ke`) and contact thickness (`particle_radius`) have different roles. Cloth density is mass per area. Resolution changes affect contact sampling as well as deformation; use `builder.color()` before finalizing the VBD scene.

VBD handles cloth self-contact and particle–shape interaction with the jaw proxies and surrounding geometry. `particle_self_contact_margin` sets interaction distance; `particle_self_contact_gap` adds detection slack. The collision pipeline's `soft_contact_gap` is a separate particle–shape parameter. Thin material must have sustained opposing contacts during lifting and transport; a visually closed gripper alone proves nothing.

### 2. Add the cable

At `TODO Step 2`, prepare `newton.Rod(points=positions, radius=...)` and pass it to `builder.add_rod(rod=rod, body_frame_origin="com", ...)`. The centerline becomes capsule bodies joined by ROD joints. Record the body, joint and shape IDs for solver ownership and whole-object measurements.

The reference uses a short curved rod with a 14 mm diameter and an accessible grasp segment. Keep segment lengths large enough to avoid initial overlap between capsule ends that are not collision-filtered. The rest must remain free to bend and follow under gravity. Plan clearance for its entire extent during transport; a cable midpoint above the box does not establish successful placement.

Stretch and bend stiffness/damping are authored separately. Direct per-joint coefficients do not make the response invariant to resampling. `newton.Rod` also supports material/section data; that is a separate authoring choice to validate. The cable bodies and joints belong to the same VBD entry as the cloth.

### 3. Couple the solvers

At `TODO Step 3`, construct two `SolverCoupled.Entry` records and a `SolverCoupledProxy`.

| Entry | Owns | Role |
|-------|------|------|
| `rigid` | Robot, gripper and both cubes | MuJoCo articulation, servo dynamics and rigid cube grasps |
| `deformable` | Cable bodies/rod joints and cloth particles | VBD material response |

Ownership must be disjoint. Both jaws participate in the proxy boundary. Cube proxies let the materials respond when they touch a MuJoCo-owned cube, while feedback returns to the source solver. Proxy visibility does not give VBD a second copy of cube ownership.

Use the supplied contact-pair factory so VBD resolves its material interactions without independently resolving robot/cube contacts already handled by MuJoCo. Static table and bin shapes are visible to the relevant solver views. Native MuJoCo rigid contacts use separately authored contact response; VBD stiffness and damping are not MuJoCo time constants. Keep those material settings distinct.

Do **not** enable `integrate_with_external_rigid_solver=True` for this VBD entry: that would prevent integration of the cable bodies it owns. Select `rigid_compliant_alm=True` for VBD rigid contacts/joints; finite authored stiffness sets their compliance. Cloth particle contacts use their separate contact path. Inspect the annotated function in `solutions/clean_step_03_build_coupled_solver.py` for the complete configuration.

## Inspect the scene before stepping

Check real topology and initial payload bounds. Counting particles proves a garment was built, but does not prove that a solver is advancing it or that a task succeeds.


In [ ]:
sys.path.insert(0, str(PART))
if REFERENCE:
    sys.path.insert(0, str(PART / "solutions"))
    from clean_table_scene_solution import build_scene
else:
    from clean_table_scene import build_scene
scene = build_scene(ROBOT, device="cpu")
initial_state = scene.model.state()
newton.eval_fk(scene.model, scene.model.joint_q, scene.model.joint_qd, initial_state)
print("Bodies:", scene.model.body_count, "cloth particles:", scene.model.particle_count)
print("Cable capsules:", len(scene.cable_bodies), "cable joints:", len(scene.cable_joints))
rigid = set(scene.rigid_bodies)
cable = set(scene.cable_bodies)
cube_bodies = {int(scene.model.shape_body.numpy()[shape]) for shape in scene.cube_shapes.values()}
assert not rigid.intersection(cable)
assert rigid.union(cable) == set(range(scene.model.body_count))
assert cube_bodies <= rigid and set(scene.clutter_bodies) == cable
assert len(scene.jaw_bodies) == 2 and all(scene.jaw_shapes)
assert set(scene.jaw_bodies).union(cube_bodies) <= set(scene.proxy_bodies)
for name, points in scene.payload_points(initial_state).items():
    assert np.isfinite(points).all()
    assert not scene.bin.contains(points), f"{name} must start outside"
    print(name, "initial center:", np.round(points.mean(axis=0), 3))

### 4. Advance one coupled substep

Complete `simulate` in `clean_the_table.py`. For every substep:

1. Clear the public external-force buffers. The coupler retains and reinjects its own feedback.
2. Detect Newton contacts for the current state.
3. Call **one** coupled `step(state_0, state_1, control, contacts, sim_dt)`.
4. Synchronize the maximal-coordinate bodies back to joint coordinates with `newton.eval_ik`. This conversion is not the target optimizer used by the controller.
5. Swap the two states.
6. Call the supplied `_sample_contacts()` helper to collect the actual jaw contacts and forces at this substep. Sampling only once per controller frame can miss a brief contact or a loss of loading.

The Proxy wrapper calls the sub-solvers, synchronizes proxies, harvests reactions and reconciles only owned outputs. Its inner coupling iterations repeat the **same time interval**; do not add elapsed time for each iteration.

This reference is deliberately uncaptured. CUDA graph capture is an optimization to validate after correctness, not a reason to mix host work or native MuJoCo-C calls into an unverified graph.


## Design the grasp and transport

The controller chooses one object, approaches its grasp region with open jaws, closes, lifts, carries it over the box, opens, and withdraws. Cube grasps and material grasps can need different contact locations and orientations. The scratch `GripperIK` helper converts those requests into joint targets, using bounded least squares for material grasps and reBot. It constrains the closing-axis orientation and rejects unreachable position targets. The dynamic Newton scene determines whether the grasp holds.

The reference handles the red cube, blue cube, cable and shirt in that order. Earlier placements can remove obstacles to later approaches, so an isolated material probe with cubes still on the table is a different scene interaction. The reBot controller adds a robot-only gravity estimate to the arm drive targets while retaining actuator limits and the imported finger gains. During SO-101 cloth release, it smoothly turns the opening wrist so fabric draped over a finger can fall free; only the robot is commanded, never a cloth particle.

The measurement helpers read the real jaw contacts, forces, aperture, payload bounds and motion. They must observe a sustained loaded grasp and an airborne carry before accepting release. Merely reaching the next controller phase cannot make a missed or dropped object count as a successful pickup.

This is a scripted task with a particular scene and material preparation, not a general-purpose cleanup policy.

## Run the selected implementation

Request a headless checked rollout first. The shared report gate requires every object to be grasped, fully lifted, carried above the box, intentionally released, and settled while detached from both jaws. It checks whole geometry, including cube corners, cloth particle radii and capsule extents.

The current runner allows up to **5000 frames** (100 simulated seconds at 50 Hz) and stops early only when the measured task is complete. A budget expiration fails. The final gripper must be withdrawn: `tool_clearance_m` is a measured jaw clearance, and `tool_withdrawn` requires at least 0.02 m above the table/rim reference.

The cell uses CPU explicitly and a fresh output directory, then applies the same schema-2 report validator as the final-check notebook. It records the trajectory so the report and movement can be inspected together. To test CUDA separately, select `cuda:0` or `cuda:1` in the command and expected device together. One process uses one device.

In [ ]:
import hashlib
import tempfile
from final_check import validate_clean_table_report

mode = "reference" if REFERENCE else "student"
artifact_root = PART.resolve() / ".generated"
artifact_root.mkdir(exist_ok=True)
run_directory = Path(tempfile.mkdtemp(prefix=f"notebook_{mode}_gripper_{ROBOT}_", dir=artifact_root))
report_path, record_path = run_directory / "report.json", run_directory / "trajectory.npz"
script = Path("solutions/clean_the_table_solution.py") if REFERENCE else Path("clean_the_table.py")
command = [sys.executable, str(script), "--robot", ROBOT, "--device", "cpu", "--viewer", "null", "--test",
           "--report", str(report_path), "--record", str(record_path)]
completed = subprocess.run(command, cwd=PART, text=True, capture_output=True, timeout=1800)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
completed.check_returncode()
result = json.loads(report_path.read_text())
assert result["newton"] == "1.6.0" and result["robot_asset_format"] == "USD"
ok, detail = validate_clean_table_report(result, robot=ROBOT, device="cpu")
assert ok, detail
assert result["record_sha256"] == hashlib.sha256(record_path.read_bytes()).hexdigest()
with np.load(record_path, allow_pickle=False) as trace:
    assert len(trace["phase"]) == len(trace["active_object"])
    assert (trace["phase"] == "carry").any(), "No carry phase was recorded"
    assert set(result["objects"]) <= set(trace["active_object"]), "An object is absent from the trajectory"
print(detail)
print("Saved report and trajectory:", run_directory)

## Inspect the measured result

Read each object's event times as well as its final speed. Grasp must precede lift and carry; the opening command must precede release. The plot and values below come from the report just validated, with no prefilled success or timing data.

In [ ]:
import matplotlib.pyplot as plt

names = list(result["objects"])
speeds = [result["objects"][name]["max_point_speed"] for name in names]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(names, speeds, color=["#db4740", "#397ad6", "#e1a23a", "#43a880"])
ax.axhline(0.04, color="black", linestyle="--", label="settling speed limit")
ax.set_ylabel("Maximum point speed (m/s)")
ax.set_title(f"Measured final gripper task: {ROBOT}, {result['device']}")
ax.legend()
plt.show()
for name in names:
    obj = result["objects"][name]
    print(name, {event: obj[event] for event in (
        "grasp_time", "lift_time", "carry_time", "release_command_time", "release_time")})
print("Measured jaw clearance:", result["tool_clearance_m"], "withdrawn:", result["tool_withdrawn"])
print("Measured coupling-input force norm maximum:", result["max_coupling_input_force_norm"])

## Watch it run

From `newton/part3`, run `PXR_WORK_THREAD_LIMIT=1 python solutions/clean_the_table_solution.py --robot so101 --device cpu --viewer gl --test` on a local desktop. Change the profile to `rebot` as needed. The viewer reads Newton states; the fabric shown is the simulated mesh. Checked runs stop at measured completion or the frame budget; an interactive window closing is not a successful task check.

The previous cell saves a real trajectory and matching JSON. `record_sha256` binds the exact NPZ bytes; `phase` and `active_object` identify the controller segment and selected payload in each recorded frame. `joint_drive_target` records the targets actually sent to the servos, including feedforward, while `gripper_command` records nominal IK commands. Use the [recording exporter](../tools/render_clean_table.py) only after the current report passes the shared validator. The default renderer plots collision geometry; `--renderer gl` replays matching official robot meshes with a model/version check. Neither advances physics. the final-check notebook saves its pairs in `.generated/final_check/`, with separate reference/student and explicit CUDA-device names.

macOS runs this physics on CPU. CUDA execution and performance require their own device runs; a plotted or rendered trajectory is not a throughput measurement.

## Troubleshooting

| Symptom | Check first |
|---------|-------------|
| USD preflight asks for Work limit 1 | Relaunch Python/Jupyter with `PXR_WORK_THREAD_LIMIT=1` and restart the kernel. |
| Cache validation fails | Keep the existing files and select a fresh automatic-cache directory; see the setup guide for intentional path overrides. |
| Cloth or cable is not lifted | Grasp location, jaw geometry, both contact forces, material resolution and full-object clearance |
| Payload falls during transport | Sustained opposing contacts and the recorded aperture; an eventual landing inside the box does not repair a lost grasp. |
| Cable does not move | Cable bodies/joints must belong to VBD, with their integration enabled. |
| Objects jump or buffers overflow | Initial penetration, contact geometry, timestep, iterations and capacity checks |
| Center is inside but validation fails | Inspect complete geometry, detached settling and the ordered grasp/lift/carry/release events. |
| Report is missing or incomplete | Read the task traceback and confirm the selected reference/student entry point. |

Do not loosen the task gate to hide a failed pickup or keep trusting a run after contact-buffer overflow.

## Recap

- Why do MuJoCo and VBD each need an explicit ownership set?
- How can VBD see a cube proxy while MuJoCo remains its owner?
- Why must both jaws participate in a material grasp?
- Which measurements distinguish a carry from dragging or a dropped payload?
- Why is checking only the cable midpoint insufficient?

## Next

Run [Final check](04__final_check.ipynb) to check the migration and all-object gripper task for both robot profiles. It reports failures and unrun CUDA checks explicitly.

## References

- [Newton 1.6.0 changelog](https://github.com/newton-physics/newton/blob/v1.6.0/CHANGELOG.md)
- [Coupled solvers](https://newton-physics.github.io/newton/stable/concepts/coupling.html)
- [MuJoCo solver guide](https://newton-physics.github.io/newton/stable/solvers/mujoco.html)
- [Solver feature matrix](https://newton-physics.github.io/newton/stable/solvers/index.html)
- [Prepared Rod implementation](https://github.com/newton-physics/newton/blob/v1.6.0/newton/_src/sim/rod.py)
- [Pinned official robot assets](https://github.com/newton-physics/newton-assets/tree/f8fb7abcbeba2318814a74f3eeb02780ad7925d6)
- [Official robot-and-cable example](https://github.com/newton-physics/newton/blob/v1.6.0/newton/examples/multiphysics/example_franka_cable_ik_pick_place.py)

The stable documentation follows the current stable release; use the pinned source when checking the exact companion API.